In [73]:
import pandas as pd

df_sch_info = pd.read_csv('General information of schools.csv')
df_subjects = pd.read_csv('Subjects Offered.csv')
df_distprogrammes = pd.read_csv('School Distinctive Programmes.csv')
df_ccas = pd.read_csv('Co-curricular activities (CCAs).csv')

In [74]:
import openpyxl

df_affiliations = pd.read_excel('school_affiliations_psle.xlsx', sheet_name=0)
df_psle = pd.read_excel('school_affiliations_psle.xlsx', sheet_name=1)

print(df_affiliations)
print(df_psle.head())

                                     school_name  \
0                  Anglo-Chinese School (Junior)   
1                  Anglo-Chinese School (Junior)   
2                 Anglo-Chinese School (Primary)   
3                 Anglo-Chinese School (Primary)   
4                           Catholic High School   
5                             De La Salle School   
6                             De La Salle School   
7                       Maris Stella High School   
8                         Montfort Junior School   
9                     St. Andrew's Junior School   
10                  St. Anthony's Primary School   
11                  St. Anthony's Primary School   
12                  St. Gabriel's Primary School   
13               St. Joseph's Institution Junior   
14               St. Joseph's Institution Junior   
15                          St. Stephen's School   
16                          St. Stephen's School   
17                Canossa Convent Primary School   
18          

In [75]:
import requests
import pandas as pd

years = ['2022', '2023', '2024', '2025']
phases = ['1', '2A', '2B', '2C', '2CS']
all_data = []

for year in years:
    for phase in phases:
        url = f"https://elite.com.sg/data/{year}-{phase}.json"
        response = requests.get(url)
        
        if response.status_code != 200:
            print(f"✗ {year} Phase {phase} — not found")
            continue
            
        df = pd.DataFrame(response.json())
        df = df.explode('school_list').reset_index(drop=True)
        df_schools_json = pd.json_normalize(df['school_list'])
        df_final = pd.concat([df.drop(columns=['school_list']), df_schools_json], axis=1)
        df_final['phase'] = phase
        df_final['year'] = year
        all_data.append(df_final)

df_ballots = pd.concat(all_data, ignore_index=True)
print(df_ballots.head())

      school_type                                        school_area  ID  \
0  primary_school  [{'id': 1, 'name': 'All Areas'}, {'id': 2, 'na...   1   
1  primary_school  [{'id': 1, 'name': 'All Areas'}, {'id': 2, 'na...   2   
2  primary_school  [{'id': 1, 'name': 'All Areas'}, {'id': 2, 'na...   3   
3  primary_school  [{'id': 1, 'name': 'All Areas'}, {'id': 2, 'na...   4   
4  primary_school  [{'id': 1, 'name': 'All Areas'}, {'id': 2, 'na...   5   

                          title total avail max_vacancies applicant  active  \
0      Admiralty Primary School     0   150                     107    True   
1  Ahmad Ibrahim Primary School     0   130                      56    True   
2                Ai Tong School     0   240                     130    True   
3      Alexandra Primary School     0   140                      75    True   
4   Anchor Green Primary School     0   180                      75    True   

  ballot remark                                findurl         area 

In [76]:
df_ballots = df_ballots[['year', 'phase', 'title', 'avail', 'applicant', 'active']]
df_ballots = df_ballots.rename(columns={
    'title': 'school_name',
    'avail': 'vacancies',
    'applicant': 'applicants'
})
df_ballots['applicants'] = pd.to_numeric(df_ballots['applicants'], errors='coerce')
df_ballots['vacancies'] = pd.to_numeric(df_ballots['vacancies'], errors='coerce')
print(df_ballots)

      year phase                   school_name  vacancies  applicants  active
0     2022     1      Admiralty Primary School      150.0       107.0    True
1     2022     1  Ahmad Ibrahim Primary School      130.0        56.0    True
2     2022     1                Ai Tong School      240.0       130.0    True
3     2022     1      Alexandra Primary School      140.0        75.0    True
4     2022     1   Anchor Green Primary School      180.0        75.0    True
...    ...   ...                           ...        ...         ...     ...
3600  2025   2CS          Yuhua Primary School      148.0        17.0    True
3601  2025   2CS          Yumin Primary School      191.0        74.0    True
3602  2025   2CS        Zhangde Primary School       13.0        19.0    True
3603  2025   2CS       Zhenghua Primary School       12.0         8.0    True
3604  2025   2CS       Zhonghua Primary School      125.0        36.0    True

[3605 rows x 6 columns]


In [77]:
#Demand score calculation for each phase and take an average of all years (2022-2025), where demand score = (applicants / vacancies)
# Filter active only and calculate demand score
df_ballots = df_ballots[df_ballots['active'] == True].copy()
df_ballots = df_ballots.groupby(['school_name', 'phase'])[['vacancies', 'applicants']].mean().reset_index()
df_ballots['demand_score'] = df_ballots['applicants'] / df_ballots['vacancies']

# Pivot to wide format
phase_map = {'1': 'P1', '2A': 'P2A', '2B': 'P2B', '2C': 'P2C', '2CS': 'P2CS'}
df_ballots['phase'] = df_ballots['phase'].map(phase_map)

df_demand = df_ballots.pivot_table(
    index='school_name', 
    columns='phase', 
    values='demand_score',
).reset_index()

# Rename columns
df_demand = df_demand.rename(columns={
    'P1': 'P1_demand',
    'P2A': 'P2A_demand',
    'P2B': 'P2B_demand',
    'P2C': 'P2C_demand',
    'P2CS': 'P2CS_demand'
})
df_demand = df_demand.round(2)
#School faced with no demand in P2CS is likely due to high demand in P2C, so we can set it to 0 instead of NaN
df_demand['P2CS_demand'] = df_demand['P2CS_demand'].fillna(0) 

df_demand['school_name'] = df_demand['school_name'].str.upper()
df_demand['school_name'] = df_demand['school_name'].replace({
    "MARIS STELLA HIGH SCHOOL (PRIMARY SECTION)": 'MARIS STELLA HIGH SCHOOL',
    "CHIJ ST. NICHOLAS GIRLS' SCHOOL (PRIMARY SECTION)": "CHIJ ST. NICHOLAS GIRLS' SCHOOL",
    "CATHOLIC HIGH SCHOOL (PRIMARY SECTION)": "CATHOLIC HIGH SCHOOL"
})

df_demand.to_csv('school_demand_scores.csv', index=False)
print(df_demand.head())

phase                   school_name  P1_demand  P2A_demand  P2B_demand  \
0          ADMIRALTY PRIMARY SCHOOL       0.59        0.59        1.22   
1      AHMAD IBRAHIM PRIMARY SCHOOL       0.30        0.06        0.00   
2                    AI TONG SCHOOL       0.53        1.19        2.67   
3          ALEXANDRA PRIMARY SCHOOL       0.48        0.53        0.13   
4       ANCHOR GREEN PRIMARY SCHOOL       0.39        0.29        0.00   

phase  P2C_demand  P2CS_demand  
0            1.84         0.00  
1            0.18         0.17  
2            1.98         0.00  
3            1.21         0.00  
4            0.20         0.67  


In [78]:
df_sch_info = df_sch_info[df_sch_info['mainlevel_code'].isin(['PRIMARY', 'MIXED LEVEL (P1-S4)'])]
df_sch_info = df_sch_info[['school_name', 'address', 'postal_code', 'nature_code', 'sap_ind', 'autonomous_ind', 'gifted_ind']]
df_sch_info = df_sch_info.replace({'na': 0, 'Yes': 1, 'No': 0})
df_sch_info['school_name'] = df_sch_info['school_name'].replace(
    "ST ANDREW'S SCHOOL (JUNIOR)", 
    "ST. ANDREW'S JUNIOR SCHOOL"
)
print(df_sch_info)

                      school_name                        address  postal_code  \
0        ADMIRALTY PRIMARY SCHOOL         11 WOODLANDS CIRCLE          738907   
2    AHMAD IBRAHIM PRIMARY SCHOOL         10 YISHUN STREET 11          768643   
4                  AI TONG SCHOOL       100 Bright Hill Drive          579646   
5        ALEXANDRA PRIMARY SCHOOL  2A Prince Charles Crescent          159016   
6     ANCHOR GREEN PRIMARY SCHOOL         31 Anchorvale Drive          544969   
..                            ...                            ...          ...   
327          YUHUA PRIMARY SCHOOL   158 JURONG EAST STREET 24          609558   
329          YUMIN PRIMARY SCHOOL        3 TAMPINES STREET 21          529393   
332        ZHANGDE PRIMARY SCHOOL            51 Jalan Membina          169485   
333       ZHENGHUA PRIMARY SCHOOL                9 Fajar Road          679002   
335       ZHONGHUA PRIMARY SCHOOL       12 SERANGOON AVENUE 4          556095   

      nature_code  sap_ind 

/var/folders/2s/sj7n8qbs5838w7lml1xsqlv40000gn/T/ipykernel_11461/1684637911.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_sch_info = df_sch_info.replace({'na': 0, 'Yes': 1, 'No': 0})


In [79]:
import re
from rapidfuzz import process, fuzz

# clean school names for standaridisation
def clean_school_name(name):
    # Remove bracketed suffixes like (PRIMARY), (PRIMARY SECTION), (JUNIOR)
    name = re.sub(r'\s*\(.*?\)', '', str(name)).strip()
    return name.upper()

# filter secondary schools
def is_secondary(name):
    secondary_keywords = ['SECONDARY', 'HIGH SCHOOL', 'JUNIOR COLLEGE', 'JC', 'INSTITUTE']
    return any(keyword in name.upper() for keyword in secondary_keywords)

# fuzzy match
def fuzzy_match_schools(school_name, sch_info_names, threshold):
    school_name = clean_school_name(school_name)
    
    # Exact match first
    sch_info_upper = [n.upper() for n in sch_info_names]
    if school_name in sch_info_upper:
        return sch_info_names[sch_info_upper.index(school_name)]
    
    # partial_ratio match
    match, score, _ = process.extractOne(
        school_name,
        sch_info_names,
        scorer=fuzz.partial_ratio
    )
    if score >= threshold:
        return match

    # Fallback: strip suffixes and retry
    suffixes = ["PRIMARY SCHOOL", "SCHOOL", "PRIMARY"]
    stripped_names = {}
    for name in sch_info_names:
        stripped = name.upper()
        for suffix in suffixes:
            if stripped.endswith(suffix):
                stripped = stripped[:-len(suffix)].strip()
                break
        stripped_names[stripped] = name

    match, score, _ = process.extractOne(
        school_name,
        list(stripped_names.keys()),
        scorer=fuzz.partial_ratio
    )
    if score >= threshold:
        return stripped_names[match]
    
    return None

In [80]:
# apply fuzzy matching to map df_distprogrammes schools to df_sch_info
df_distprogrammes = df_distprogrammes[~df_distprogrammes['school_name'].apply(is_secondary)].copy()
df_distprogrammes['school_name_matched'] = df_distprogrammes['school_name'].apply(
    lambda x: fuzzy_match_schools(x, df_sch_info['school_name'].values, 90)
)
df_distprogrammes = df_distprogrammes[['school_name_matched', 'alp_domain']]
df_distprogrammes = df_distprogrammes.rename(columns={'school_name_matched': 'school_name'})

df_distprogrammes_pivot = df_distprogrammes.groupby('school_name').size().reset_index(name='distprog_count')
print(df_distprogrammes_pivot)

# clean and filter
df_distprogrammes['school_name'] = df_distprogrammes['school_name'].apply(
    lambda x: fuzzy_match_schools(x, df_sch_info['school_name'].tolist(), threshold=80)
)

df_distprogrammes_pivot = df_distprogrammes.groupby('school_name').size().reset_index(name='distprog_count')
print(df_distprogrammes_pivot.head())


                      school_name  distprog_count
0        ADMIRALTY PRIMARY SCHOOL               1
1    AHMAD IBRAHIM PRIMARY SCHOOL               1
2                  AI TONG SCHOOL               1
3        ALEXANDRA PRIMARY SCHOOL               1
4     ANCHOR GREEN PRIMARY SCHOOL               1
..                            ...             ...
173          YUHUA PRIMARY SCHOOL               1
174          YUMIN PRIMARY SCHOOL               1
175        ZHANGDE PRIMARY SCHOOL               1
176       ZHENGHUA PRIMARY SCHOOL               1
177       ZHONGHUA PRIMARY SCHOOL               1

[178 rows x 2 columns]
                    school_name  distprog_count
0      ADMIRALTY PRIMARY SCHOOL               1
1  AHMAD IBRAHIM PRIMARY SCHOOL               1
2                AI TONG SCHOOL               1
3      ALEXANDRA PRIMARY SCHOOL               1
4   ANCHOR GREEN PRIMARY SCHOOL               1


In [ ]:
# Filter to include only schools from df_sch_info
df_subjects = df_subjects.rename(columns={'School_Name': 'school_name'}) 
df_subjects = df_subjects[df_subjects['school_name'].isin(df_sch_info['school_name'])]
df_ccas = df_ccas.rename(columns={'School_name': 'school_name'})
df_ccas = df_ccas[df_ccas['school_name'].isin(df_sch_info['school_name'])]

print(df_ccas.head())
print(df_subjects.head())

                school_name school_section  \
0  ADMIRALTY PRIMARY SCHOOL        PRIMARY   
1  ADMIRALTY PRIMARY SCHOOL        PRIMARY   
2  ADMIRALTY PRIMARY SCHOOL        PRIMARY   
3  ADMIRALTY PRIMARY SCHOOL        PRIMARY   
4  ADMIRALTY PRIMARY SCHOOL        PRIMARY   

                      cca_grouping_desc            cca_generic_name  \
0                        ART AND CRAFTS         CLUBS AND SOCIETIES   
1                         CHINESE DANCE  VISUAL AND PERFORMING ARTS   
2                                 CHOIR  VISUAL AND PERFORMING ARTS   
3                 DESIGN AND INNOVATION         CLUBS AND SOCIETIES   
4  ENGLISH LANGUAGE, DRAMA AND DEBATING         CLUBS AND SOCIETIES   

  cca_customized_name  
0                  na  
1                  na  
2                  na  
3                  na  
4                  na  
                school_name                 Subject_Desc
0  ADMIRALTY PRIMARY SCHOOL                          Art
1  ADMIRALTY PRIMARY SCHOOL           

In [82]:
# Keep only relevant columns for df_ccas
df_ccas = df_ccas[['school_name', 'cca_grouping_desc', 'cca_generic_name']]

print(df_ccas.head())

                school_name                     cca_grouping_desc  \
0  ADMIRALTY PRIMARY SCHOOL                        ART AND CRAFTS   
1  ADMIRALTY PRIMARY SCHOOL                         CHINESE DANCE   
2  ADMIRALTY PRIMARY SCHOOL                                 CHOIR   
3  ADMIRALTY PRIMARY SCHOOL                 DESIGN AND INNOVATION   
4  ADMIRALTY PRIMARY SCHOOL  ENGLISH LANGUAGE, DRAMA AND DEBATING   

             cca_generic_name  
0         CLUBS AND SOCIETIES  
1  VISUAL AND PERFORMING ARTS  
2  VISUAL AND PERFORMING ARTS  
3         CLUBS AND SOCIETIES  
4         CLUBS AND SOCIETIES  


In [83]:
# Pivot df_ccas to create a counter of each cca_generic_name per school
df_ccas_pivot = df_ccas.groupby(['school_name', 'cca_generic_name']).size().unstack(fill_value=0)
df_ccas_pivot = df_ccas_pivot.reset_index()
df_ccas_pivot.columns = ['school_name', 'cca_clubs', 'cca_others', 'cca_sports', 'cca_uniformed', 'cca_arts']
print(df_ccas_pivot.head())

                    school_name  cca_clubs  cca_others  cca_sports  \
0      ADMIRALTY PRIMARY SCHOOL          5           0           5   
1  AHMAD IBRAHIM PRIMARY SCHOOL          3           1           4   
2                AI TONG SCHOOL          4           0           6   
3      ALEXANDRA PRIMARY SCHOOL          6           0           7   
4   ANCHOR GREEN PRIMARY SCHOOL          4           0           4   

   cca_uniformed  cca_arts  
0              2         6  
1              1         4  
2              2         5  
3              1         5  
4              1         5  


In [84]:
# Pivot df_subjects to create a counter of each subject per school
df_subjects_pivot = df_subjects.groupby('school_name').size().reset_index(name='subject_count')

print(df_subjects_pivot)

                      school_name  subject_count
0        ADMIRALTY PRIMARY SCHOOL             24
1    AHMAD IBRAHIM PRIMARY SCHOOL             21
2                  AI TONG SCHOOL             14
3        ALEXANDRA PRIMARY SCHOOL             22
4     ANCHOR GREEN PRIMARY SCHOOL             30
..                            ...            ...
177          YUHUA PRIMARY SCHOOL             23
178          YUMIN PRIMARY SCHOOL             23
179        ZHANGDE PRIMARY SCHOOL             29
180       ZHENGHUA PRIMARY SCHOOL             25
181       ZHONGHUA PRIMARY SCHOOL             23

[182 rows x 2 columns]


In [85]:
df_affiliations = df_affiliations.copy()
df_affiliations['school_name'] = df_affiliations['school_name'].str.upper()
# Pivot df_affiliations to create a counter of affiliations per school
df_affiliations_pivot = df_affiliations.groupby('school_name').size().reset_index(name='affiliation_count')

df_affiliations_pivot

,school_name,affiliation_count
0,ANGLO-CHINESE SCHOOL (JUNIOR),2
1,ANGLO-CHINESE SCHOOL (PRIMARY),2
2,CANOSSA CONVENT PRIMARY SCHOOL,1
3,CATHOLIC HIGH SCHOOL,1
4,CHIJ (KATONG) PRIMARY,1
5,CHIJ (KELLOCK),1
6,CHIJ OUR LADY OF GOOD COUNSEL,1
7,CHIJ OUR LADY OF THE NATIVITY,1
8,CHIJ OUR LADY QUEEN OF PEACE,1
9,CHIJ PRIMARY (TOA PAYOH),1


In [86]:
df_psle['school_name'] = df_psle['school_name'].str.upper()
print(df_psle)

   rank                              school_name median_psle_score_range
0     1            RAFFLES GIRLS' PRIMARY SCHOOL                     4-7
1     2                   NANYANG PRIMARY SCHOOL                     4-7
2     3                     CATHOLIC HIGH SCHOOL                     5-8
3     4           ANGLO-CHINESE SCHOOL (PRIMARY)                     5-8
4     5                HENRY PARK PRIMARY SCHOOL                     5-8
5     6                            ROSYTH SCHOOL                     5-8
6     7                           AI TONG SCHOOL                     6-9
7     8        METHODIST GIRLS' SCHOOL (PRIMARY)                     6-9
8     9  SINGAPORE CHINESE GIRLS' PRIMARY SCHOOL                     6-9
9    10          CHIJ ST. NICHOLAS GIRLS' SCHOOL                     6-9


In [87]:
# Merge all datasets with df_sch_info as the base
df_schools = df_sch_info.copy()

# Merge df_demand
df_schools = df_schools.merge(df_demand, left_on='school_name', right_on='school_name', how='left')

# Merge df_subjects_pivot
df_schools = df_schools.merge(df_subjects_pivot, left_on='school_name', right_on='school_name', how='left')

# Merge df_distprogrammes_pivot
df_schools = df_schools.merge(df_distprogrammes_pivot, left_on='school_name', right_on='school_name', how='left')

# Merge df_ccas_pivot
df_schools = df_schools.merge(df_ccas_pivot, left_on='school_name', right_on='school_name', how='left')

# Merge df_affiliations_pivot
df_schools = df_schools.merge(df_affiliations_pivot, left_on='school_name', right_on='school_name', how='left')

# Add top_psle_score column
df_schools['top_psle_score'] = df_schools['school_name'].isin(df_psle['school_name']).astype(int)

# if NaN replace with 0
df_schools = df_schools.fillna(0)

print(df_schools)

                      school_name                        address  postal_code  \
0        ADMIRALTY PRIMARY SCHOOL         11 WOODLANDS CIRCLE          738907   
1    AHMAD IBRAHIM PRIMARY SCHOOL         10 YISHUN STREET 11          768643   
2                  AI TONG SCHOOL       100 Bright Hill Drive          579646   
3        ALEXANDRA PRIMARY SCHOOL  2A Prince Charles Crescent          159016   
4     ANCHOR GREEN PRIMARY SCHOOL         31 Anchorvale Drive          544969   
..                            ...                            ...          ...   
177          YUHUA PRIMARY SCHOOL   158 JURONG EAST STREET 24          609558   
178          YUMIN PRIMARY SCHOOL        3 TAMPINES STREET 21          529393   
179        ZHANGDE PRIMARY SCHOOL            51 Jalan Membina          169485   
180       ZHENGHUA PRIMARY SCHOOL                9 Fajar Road          679002   
181       ZHONGHUA PRIMARY SCHOOL       12 SERANGOON AVENUE 4          556095   

      nature_code  sap_ind 

In [88]:
# Convert numeric columns to integer, excluding Popularity and Admission
numeric_cols = df_schools.select_dtypes(include=['float64', 'int64']).columns
cols_to_convert = [col for col in numeric_cols if col not in ['P1_demand', 'P2A_demand', 'P2B_demand', 'P2C_demand', 'P2CS_demand']]

for col in cols_to_convert:
    df_schools[col] = df_schools[col].astype('int64')

print(df_schools)

                      school_name                        address  postal_code  \
0        ADMIRALTY PRIMARY SCHOOL         11 WOODLANDS CIRCLE          738907   
1    AHMAD IBRAHIM PRIMARY SCHOOL         10 YISHUN STREET 11          768643   
2                  AI TONG SCHOOL       100 Bright Hill Drive          579646   
3        ALEXANDRA PRIMARY SCHOOL  2A Prince Charles Crescent          159016   
4     ANCHOR GREEN PRIMARY SCHOOL         31 Anchorvale Drive          544969   
..                            ...                            ...          ...   
177          YUHUA PRIMARY SCHOOL   158 JURONG EAST STREET 24          609558   
178          YUMIN PRIMARY SCHOOL        3 TAMPINES STREET 21          529393   
179        ZHANGDE PRIMARY SCHOOL            51 Jalan Membina          169485   
180       ZHENGHUA PRIMARY SCHOOL                9 Fajar Road          679002   
181       ZHONGHUA PRIMARY SCHOOL       12 SERANGOON AVENUE 4          556095   

      nature_code  sap_ind 

In [89]:
df_schools.to_csv('../school_scoring/schools_data.csv', index=False)

In [90]:
df_schools

,school_name,address,postal_code,nature_code,sap_ind,autonomous_ind,gifted_ind,P1_demand,P2A_demand,P2B_demand,...,P2CS_demand,subject_count,distprog_count,cca_clubs,cca_others,cca_sports,cca_uniformed,cca_arts,affiliation_count,top_psle_score
0,ADMIRALTY PRIMARY SCHOOL,11 WOODLANDS CIRCLE,738907,CO-ED SCHOOL,0,0,0,0.59,0.59,1.22,...,0.00,24,1,5,0,5,2,6,0,0
1,AHMAD IBRAHIM PRIMARY SCHOOL,10 YISHUN STREET 11,768643,CO-ED SCHOOL,0,0,0,0.30,0.06,0.00,...,0.17,21,1,3,1,4,1,4,0,0
2,AI TONG SCHOOL,100 Bright Hill Drive,579646,CO-ED SCHOOL,1,0,0,0.53,1.19,2.67,...,0.00,14,1,4,0,6,2,5,0,1
3,ALEXANDRA PRIMARY SCHOOL,2A Prince Charles Crescent,159016,CO-ED SCHOOL,0,0,0,0.48,0.53,0.13,...,0.00,22,1,6,0,7,1,5,0,0
4,ANCHOR GREEN PRIMARY SCHOOL,31 Anchorvale Drive,544969,CO-ED SCHOOL,0,0,0,0.39,0.29,0.00,...,0.67,30,1,4,0,4,1,5,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177,YUHUA PRIMARY SCHOOL,158 JURONG EAST STREET 24,609558,CO-ED SCHOOL,0,0,0,0.29,0.11,0.01,...,0.17,23,1,4,0,7,1,6,0,0
178,YUMIN PRIMARY SCHOOL,3 TAMPINES STREET 21,529393,CO-ED SCHOOL,0,0,0,0.34,0.09,0.00,...,0.44,23,1,3,0,3,2,4,0,0
179,ZHANGDE PRIMARY SCHOOL,51 Jalan Membina,169485,CO-ED SCHOOL,0,0,0,0.58,0.33,0.02,...,1.57,29,1,1,0,8,2,2,0,0
180,ZHENGHUA PRIMARY SCHOOL,9 Fajar Road,679002,CO-ED SCHOOL,0,0,0,0.52,0.80,0.02,...,1.34,25,1,3,0,4,2,6,0,0
